# LAB 3 — Ollama: LLM Local para Analise Juridica

**Curso:** MBA RAG & CAG Aplicados a Direito e Segurança Pública  
**Aula:** 1 — Fundamentos  
**Duracao estimada:** 45 minutos  
**Ambiente:** Jupyter Local no VS Code (Windows / macOS / Linux)

---

## Objetivos de Aprendizagem
1. Conectar ao servidor Ollama via API REST e via cliente OpenAI-compativel
2. Executar o primeiro completion juridico com Llama 3.x
3. Experimentar o impacto dos parametros de geracao (temperature, max_tokens)
4. Demonstrar system messages para perfis juridicos distintos
5. Comparar Ollama com abordagens cloud (OpenAI) e entender trade-offs de privacidade

---

## Arquitetura do Lab

```
Notebook Jupyter (VS Code)
    |
    | HTTP POST /api/generate
    | HTTP POST /v1/chat/completions (OpenAI-compat)
    |
    v
Ollama (localhost:11434)
    |
    v
Modelo: llama3.2:3b ou llama3.1:8b (GGUF quantizado)
    |
    v
GPU local (se disponivel) ou CPU
```

> **PRE-REQUISITO:** Ollama instalado e rodando. Se ainda nao fez,  
> consulte o `ROTEIRO_INSTALACAO_FERRAMENTAS.md` e execute o LAB2 primeiro.

---

## Por que Ollama em vez de vLLM?

| Caracteristica | vLLM | Ollama |
|---------------|------|--------|
| Sistemas suportados | Linux (NVIDIA CUDA) | Windows, macOS, Linux |
| Instalacao | pip + CUDA toolkit | Instalador unico |
| GPU obrigatoria | Sim | Nao (roda em CPU) |
| API | OpenAI-compativel | REST nativa + OpenAI-compativel |
| Uso no curso | Legado (aulas anteriores) | Padrao atual do curso |
| Modelos | HuggingFace Hub | Biblioteca propria + HuggingFace |

Para producao em Kubernetes (Aula 12), o Ollama pode ser substituido por vLLM ou  
llama.cpp servindo via API — a interface OpenAI-compativel garante portabilidade do codigo.

## CELULA 1 — Verificar Ambiente e Instalar Dependencias

In [ ]:
# CELULA 1: Verificar ambiente e instalar dependencias
import subprocess, sys, requests

# Instalar dependencias se nao estiverem presentes
PACOTES = ['openai>=1.40.0', 'requests']
for pacote in PACOTES:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pacote, '--quiet'],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f'ERRO ao instalar {pacote}: {result.stderr[:100]}')

# Verificar GPU
print('VERIFICACAO DE HARDWARE')
print('=' * 60)
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.returncode == 0:
    for linha in result.stdout.strip().split('\n'):
        print(f'GPU: {linha}')
    print('Ollama usara a GPU automaticamente.')
else:
    print('GPU NVIDIA nao detectada — Ollama rodara em CPU.')
    print('Performance sera menor, mas todos os exemplos funcionam.')

print()

# Verificar Ollama
print('VERIFICANDO SERVIDOR OLLAMA')
print('=' * 60)

OLLAMA_BASE_URL = 'http://localhost:11434'

try:
    r = requests.get(f'{OLLAMA_BASE_URL}/api/tags', timeout=5)
    modelos = [m['name'] for m in r.json().get('models', [])]
    print(f'Ollama ONLINE')
    print(f'Modelos disponiveis: {modelos}')

    if not modelos:
        print()
        print('AVISO: Nenhum modelo instalado.')
        print('Execute no terminal:')
        print('  ollama pull llama3.2:3b')
        raise SystemExit('Instale pelo menos um modelo antes de continuar.')

except requests.exceptions.ConnectionError:
    print('ERRO: Ollama nao esta rodando!')
    print('Solucoes:')
    print('  Windows: verifique o icone do Ollama na bandeja do sistema')
    print('  Linux/macOS: execute no terminal: ollama serve')
    raise SystemExit('Inicie o Ollama antes de continuar.')

## CELULA 2 — Configurar Clientes Ollama

**O que faz:** Configura duas formas de chamar o Ollama:
1. API REST nativa (`/api/generate`) — mais simples, sem dependencia extra
2. Cliente OpenAI-compativel — mesma interface do GPT-4, codigo portavel

In [ ]:
# CELULA 2: Configurar clientes
import requests
from openai import OpenAI

OLLAMA_BASE_URL = 'http://localhost:11434'

# ─── Metodo 1: API REST nativa do Ollama ─────────────────────
def chamar_ollama(prompt: str, modelo: str = 'llama3.2:3b',
                  temperature: float = 0.2, max_tokens: int = 500) -> str:
    """
    Chama o Ollama via API REST nativa.
    Endpoint: POST /api/generate
    Documentacao: https://github.com/ollama/ollama/blob/main/docs/api.md
    """
    response = requests.post(
        f'{OLLAMA_BASE_URL}/api/generate',
        json={
            'model': modelo,
            'prompt': prompt,
            'stream': False,
            'options': {
                'temperature': temperature,
                'num_predict': max_tokens,
            }
        },
        timeout=120
    )
    response.raise_for_status()
    return response.json()['response']


# ─── Metodo 2: Cliente OpenAI-compativel ─────────────────────
# O Ollama expoe /v1/chat/completions identico ao padrao OpenAI.
# Isso significa que qualquer codigo escrito para o GPT-4 funciona
# sem alteracao — basta trocar a base_url.
client_ollama = OpenAI(
    base_url=f'{OLLAMA_BASE_URL}/v1',
    api_key='ollama'   # qualquer string funciona; o Ollama nao valida
)

# Detecta o melhor modelo disponivel
modelos_disponiveis = [m.id for m in client_ollama.models.list().data]

# Prioridade: llama3.1:8b > llama3.2:3b > qualquer outro
PREFERENCIAS = ['llama3.1:8b', 'llama3.1', 'llama3.2:3b', 'llama3.2', 'llama3']
MODEL_ID = next(
    (p for pref in PREFERENCIAS for p in modelos_disponiveis if pref in p),
    modelos_disponiveis[0] if modelos_disponiveis else None
)

if not MODEL_ID:
    raise SystemExit('Nenhum modelo Llama encontrado. Execute: ollama pull llama3.2:3b')

print(f'Modelo selecionado: {MODEL_ID}')
print(f'URL base Ollama:    {OLLAMA_BASE_URL}/v1')
print()
print('Dois clientes disponiveis:')
print('  chamar_ollama()   — API REST nativa (/api/generate)')
print('  client_ollama     — Cliente OpenAI-compativel (/v1/chat/completions)')

## CELULA 3 — Primeiro Completion: Analise Juridica

**O que faz:** Envia uma pergunta juridica ao Llama local e exibe metricas de geracao.  
**Por que:** Valida o pipeline completo e demonstra a latencia do modelo local.

In [ ]:
# CELULA 3: Primeiro completion juridico
import time

SYSTEM_PROMPT = (
    'Voce e um assistente juridico especializado em direito penal brasileiro. '
    'Responda sempre de forma tecnica e concisa, citando artigos de lei quando aplicavel. '
    'NUNCA invente precedentes, sumulas ou artigos inexistentes.'
)

USER_PROMPT = (
    'Qual e a diferenca entre os crimes de furto (art. 155 CP) '
    'e roubo (art. 157 CP)? Explique os elementos que os distinguem '
    'e cite um exemplo pratico de cada.'
)

print(f'Modelo: {MODEL_ID}')
print('=' * 60)
print(f'Pergunta: {USER_PROMPT[:80]}...')
print('Aguardando resposta...')
print()

inicio = time.time()

resposta = client_ollama.chat.completions.create(
    model=MODEL_ID,
    messages=[
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': USER_PROMPT}
    ],
    temperature=0.2,
    max_tokens=500,
)

tempo_total     = time.time() - inicio
texto_resposta  = resposta.choices[0].message.content
tokens_prompt   = resposta.usage.prompt_tokens
tokens_resposta = resposta.usage.completion_tokens
tok_por_segundo = tokens_resposta / tempo_total if tempo_total > 0 else 0

print('METRICAS DE GERACAO:')
print(f'  Tokens (prompt):    {tokens_prompt}')
print(f'  Tokens (resposta):  {tokens_resposta}')
print(f'  Tempo total:        {tempo_total:.1f}s')
print(f'  Velocidade:         {tok_por_segundo:.1f} tokens/s')
print()
print('RESPOSTA DO MODELO:')
print('-' * 60)
print(texto_resposta)
print('-' * 60)

# Tambem demonstra a API REST nativa para comparacao
print()
print('(Mesmo resultado via API REST nativa /api/generate — para referencia)')
resp_nativa = chamar_ollama(
    f'Sistema: {SYSTEM_PROMPT}\n\nPergunta: O que e peculato? Responda em 2 frases.',
    modelo=MODEL_ID,
    temperature=0.1,
    max_tokens=80
)
print(f'Resposta (API nativa): {resp_nativa[:200]}')

## CELULA 4 — Experimento: Impacto da Temperature

**O que faz:** Envia o mesmo prompt com diferentes valores de temperature e compara as respostas.  
**Por que:** Demonstra empiricamente como temperature afeta determinismo vs criatividade —  
conhecimento fundamental para calibrar sistemas RAG juridicos.

In [ ]:
# CELULA 4: Experimento de temperature
import sys

PROMPT_CLASSIFICACAO = (
    'Classifique o seguinte crime segundo o Codigo Penal Brasileiro. '
    'Responda EXATAMENTE neste formato: '
    '"Tipo: [crime]. Artigo: [numero]. Pena maxima: [pena]."\n\n'
    'FATO: "O suspeito subtraiu R$ 500,00 da bolsa da vitima enquanto ela dormia no onibus."'
)

TEMPERATURAS = [
    (0.0,  'Deterministico — maxima precisao (ideal para extracao de dados)'),
    (0.3,  'Baixo — recomendado para analise juridica'),
    (0.7,  'Medio — equilibrio criatividade/precisao'),
    (1.0,  'Alto — criativo, menos adequado para classificacao penal'),
]

print('EXPERIMENTO: IMPACTO DA TEMPERATURE NA CLASSIFICACAO PENAL')
print('=' * 65)
print(f'Prompt: {PROMPT_CLASSIFICACAO[:100]}...')
print()

for temp, descricao in TEMPERATURAS:
    print(f'Temperature = {temp} ({descricao})')
    inicio = time.time()

    resp = client_ollama.chat.completions.create(
        model=MODEL_ID,
        messages=[
            {'role': 'system', 'content': 'Voce e um assistente juridico tecnico e objetivo.'},
            {'role': 'user',   'content': PROMPT_CLASSIFICACAO}
        ],
        temperature=temp,
        max_tokens=80,
    )

    duracao = time.time() - inicio
    texto   = resp.choices[0].message.content.strip()
    print(f'  Resposta: {texto[:250]}')
    print(f'  Tempo: {duracao:.1f}s | Tokens: {resp.usage.completion_tokens}')
    print('-' * 40)
    sys.stdout.flush()

print()
print('CONCLUSAO:')
print('  Para classificacao juridica: use temperature=0.0 a 0.2')
print('  Para geracao de argumentos: use temperature=0.5 a 0.8')
print('  Para criacao de analogias: use temperature=0.7 a 1.0')

## CELULA 5 — System Messages: Roleplay Juridico

**O que faz:** Analisa o mesmo fato juridico por tres perspectivas distintas usando system messages.  
**Por que:** Demonstra como system messages permitem especializar o LLM sem fine-tuning —  
padrao fundamental para sistemas RAG multi-persona em escritorios e orgaos publicos.

In [ ]:
# CELULA 5: System messages para perfis juridicos
FATO_JURIDICO = (
    'Fato: Acusado foi preso em flagrante com 50g de cocaina dividida em '
    'embalagens individuais. Sem antecedentes criminais. Residencia fixa comprovada. '
    'Trabalho licito declarado.'
)

PERSONAS = [
    {
        'nome': 'JUIZ DE DIREITO',
        'system': (
            'Voce e um juiz de direito. Analise a necessidade de prisao preventiva '
            'versus medidas cautelares alternativas. '
            'Cite apenas art. 319 CPP e art. 312 CPP. Seja sucinto.'
        ),
        'pergunta': 'Devo decretar prisao preventiva ou aplicar medidas cautelares alternativas?'
    },
    {
        'nome': 'DELEGADO DE POLICIA',
        'system': (
            'Voce e um delegado de policia. Indique as diligencias investigativas '
            'prioritarias. Foque em cadeia de custodia e coleta de provas.'
        ),
        'pergunta': 'Quais diligencias devo determinar imediatamente para fortalecer o inquerito?'
    },
    {
        'nome': 'ADVOGADO DE DEFESA',
        'system': (
            'Voce e um advogado criminal de defesa. Identifique '
            'as teses defensivas disponiveis. Cite dispositivos legais aplicaveis.'
        ),
        'pergunta': 'Quais sao as melhores teses de defesa disponiveis para este caso?'
    },
]

print('DEMONSTRACAO: SYSTEM MESSAGES PARA PERFIS JURIDICOS')
print('=' * 65)
print(f'Fato: {FATO_JURIDICO}')
print()

for persona in PERSONAS:
    print(f"{'─' * 15}")
    print(f"PERFIL: {persona['nome']}")
    print(f"Pergunta: {persona['pergunta']}")
    print('Gerando resposta...')

    resp = client_ollama.chat.completions.create(
        model=MODEL_ID,
        messages=[
            {'role': 'system', 'content': persona['system']},
            {'role': 'user',   'content': f"{FATO_JURIDICO}\n\n{persona['pergunta']}"}
        ],
        temperature=0.2,
        max_tokens=300,
    )
    print(f"\n{resp.choices[0].message.content.strip()}\n")

## CELULA 6 — Geracao de Embedding via Ollama

**O que faz:** Demonstra como usar o Ollama para gerar embeddings — funcionalidade que sera  
usada extensivamente no pipeline RAG (Labs 4, 5 e Exemplo).

**Conexao com a Teoria:** O Topico 3 da teoria explica como embeddings contextuais capturam  
significado semantico. Esta celula torna isso concreto.

In [ ]:
# CELULA 6: Embeddings via Ollama
import numpy as np
import requests

EMBED_MODEL = 'nomic-embed-text'  # 768 dims, padrao do curso

def gerar_embedding(texto: str, modelo: str = EMBED_MODEL) -> list:
    """
    Gera embedding de um texto via Ollama.
    Endpoint: POST /api/embeddings
    """
    r = requests.post(
        f'{OLLAMA_BASE_URL}/api/embeddings',
        json={'model': modelo, 'prompt': texto},
        timeout=30
    )
    r.raise_for_status()
    return r.json()['embedding']

def similaridade_coseno(v1: list, v2: list) -> float:
    a, b = np.array(v1), np.array(v2)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print('DEMONSTRACAO: EMBEDDINGS VIA OLLAMA')
print(f'Modelo: {EMBED_MODEL}')
print('=' * 60)

# Verifica se o modelo de embedding esta instalado
modelos_ollama = [m['name'] for m in requests.get(f'{OLLAMA_BASE_URL}/api/tags').json().get('models', [])]
if not any('nomic' in m or 'embed' in m for m in modelos_ollama):
    print('AVISO: nomic-embed-text nao encontrado.')
    print('Execute: ollama pull nomic-embed-text')
    print('Usando o primeiro modelo disponivel como fallback...')
    EMBED_MODEL = modelos_ollama[0] if modelos_ollama else None

if EMBED_MODEL:
    pares = [
        ('O reu foi absolvido por falta de provas.', 'O acusado foi inocentado por insuficiencia de evidencias.'),
        ('Crime de peculato: desvio de bens publicos.', 'Estelionato: obtencao de vantagem por fraude.'),
        ('Habeas corpus para cessar constrangimento ilegal.', 'Mandado de seguranca para proteger direito liquido e certo.'),
    ]

    print(f'{'Par de Frases':<55} {'Similaridade'}')
    print('-' * 70)

    for frase_a, frase_b in pares:
        try:
            emb_a = gerar_embedding(frase_a, EMBED_MODEL)
            emb_b = gerar_embedding(frase_b, EMBED_MODEL)
            sim   = similaridade_coseno(emb_a, emb_b)
            label = f'{frase_a[:30]}... / {frase_b[:20]}...'
            print(f'{label:<55} {sim:.3f}')
        except Exception as e:
            print(f'Erro ao gerar embedding: {e}')

    print()
    print('Interpretacao:')
    print('  > 0.8 — frases semanticamente quase identicas')
    print('  0.5–0.8 — frases relacionadas no mesmo dominio')
    print('  < 0.5 — frases com pouca relacao semantica')
    print(f'\nDimensoes do embedding: {len(gerar_embedding("teste", EMBED_MODEL))}')

## CELULA 7 — Checklist de Validacao

In [ ]:
# CELULA 7: Checklist de validacao do Lab 3
import requests, time

print('CHECKLIST DE VALIDACAO — LAB 3 (Ollama + LLM Local)')
print('=' * 65)

checks = []

# Check 1: Ollama respondendo
try:
    r = requests.get(f'{OLLAMA_BASE_URL}/api/tags', timeout=5)
    checks.append(('Servidor Ollama ativo (localhost:11434)', r.status_code == 200,
                    f'HTTP {r.status_code}'))
except Exception as e:
    checks.append(('Servidor Ollama ativo (localhost:11434)', False, str(e)[:50]))

# Check 2: Modelo LLM disponivel
try:
    r = requests.get(f'{OLLAMA_BASE_URL}/v1/models', timeout=5)
    ids = [m['id'] for m in r.json().get('data', [])]
    llm_ok = any('llama' in m for m in ids)
    checks.append(('Modelo Llama disponivel', llm_ok, f'Modelos: {ids}'))
except Exception as e:
    checks.append(('Modelo Llama disponivel', False, str(e)[:50]))

# Check 3: API chat/completions funcional
try:
    r = client_ollama.chat.completions.create(
        model=MODEL_ID,
        messages=[{'role': 'user', 'content': 'Diga apenas: OK'}],
        temperature=0.0, max_tokens=10,
    )
    txt = r.choices[0].message.content.strip()
    checks.append(('API /v1/chat/completions funcional', len(txt) > 0,
                    f'Resposta: "{txt[:30]}"'))
except Exception as e:
    checks.append(('API /v1/chat/completions funcional', False, str(e)[:60]))

# Check 4: API REST nativa funcional
try:
    txt = chamar_ollama('Diga: OK', modelo=MODEL_ID, temperature=0.0, max_tokens=10)
    checks.append(('API REST nativa /api/generate funcional', len(txt) > 0,
                    f'Resposta: "{txt[:30]}"'))
except Exception as e:
    checks.append(('API REST nativa /api/generate funcional', False, str(e)[:60]))

# Check 5: Velocidade minima
try:
    t0 = time.time()
    rb = client_ollama.chat.completions.create(
        model=MODEL_ID,
        messages=[{'role': 'user', 'content': 'Liste 3 artigos do Codigo Penal.'}],
        temperature=0.1, max_tokens=80,
    )
    dt  = time.time() - t0
    tks = rb.usage.completion_tokens / dt if dt > 0 else 0
    checks.append(('Velocidade de geracao > 2 tok/s', tks > 2,
                    f'{tks:.1f} tok/s — {"OK" if tks > 2 else "lento; normal em CPU"}'))
except Exception as e:
    checks.append(('Velocidade de geracao > 2 tok/s', False, str(e)[:60]))

# Check 6: Embedding disponivel
try:
    r = requests.post(
        f'{OLLAMA_BASE_URL}/api/embeddings',
        json={'model': 'nomic-embed-text', 'prompt': 'teste'},
        timeout=30
    )
    dims = len(r.json().get('embedding', []))
    checks.append(('Embedding nomic-embed-text disponivel', dims > 0,
                    f'{dims} dimensoes'))
except Exception as e:
    checks.append(('Embedding nomic-embed-text disponivel', False,
                    'Execute: ollama pull nomic-embed-text'))

# Imprimir resultado
for nome, ok, detalhe in checks:
    icone = '[OK]' if ok else '[XX]'
    print(f'{icone} {nome}')
    print(f'   {detalhe}')

aprovados = sum(1 for _, ok, _ in checks if ok)
print(f'\n{"=" * 65}')
print(f'Resultado: {aprovados}/{len(checks)} checks aprovados')

if aprovados == len(checks):
    print()
    print('LAB 3 CONCLUIDO COM SUCESSO!')
    print('Proximo: LAB4 — LangFuse Observabilidade')
else:
    print()
    print('Verifique se o Ollama esta rodando e os modelos foram baixados.')
    print('Terminal: ollama pull llama3.2:3b && ollama pull nomic-embed-text')